# Lecture: Controllable Generation I — img2img & Inpainting

In C4-6 we generated images purely from **text**, starting from random noise. But
text alone is a blunt instrument: we cannot easily say "keep *this* composition"
or "change only *that* part". These last two notebooks of the chapter are about
**control** — guiding the generation with more than a prompt.

This first notebook covers two of the most useful editing pipelines, both built on
the same Stable Diffusion model from C4-6:

1. **img2img** — start the diffusion process from an **existing image** instead of
   pure noise. The output keeps the overall layout/colours of the input but is
   re-imagined according to the prompt. A `strength` parameter controls how much
   is changed.

2. **Inpainting** — replace only a **masked region** of an image, leaving the rest
   untouched. This is how "remove/replace this object" tools work.

The key idea connecting both to the rest of C4: diffusion is just **denoising from
some starting point**. text2img starts from pure noise; img2img starts from a *noised
version of an input image*; inpainting only denoises *inside a mask*. Same model,
different starting conditions.

> **Requirements:** GPU runtime (Colab: T4). The first cells download model
> weights (~4 GB each for the base and inpainting models).

### Install dependencies

Same pinned, mutually compatible versions as C4-6.

In [ ]:
!pip install -q diffusers==0.31.0 transformers==4.44.2 accelerate==0.34.2

### Load the base Stable Diffusion pipeline

We start with the familiar text2img pipeline from C4-6 — we will use it both to
create a starting image and, re-purposed, for img2img.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from diffusers import StableDiffusionPipeline, StableDiffusionImg2ImgPipeline

assert torch.cuda.is_available(), "This notebook requires a GPU runtime (Colab: T4)."
device = "cuda"

MODEL_ID = "stable-diffusion-v1-5/stable-diffusion-v1-5"

pipe_t2i = StableDiffusionPipeline.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, safety_checker=None,
).to(device)

print("Base Stable Diffusion loaded.")

### Step 0 — Create a starting image

To keep everything self-contained, we **generate** the image we will later edit,
using plain text2img. Fix the seed so the rest of the notebook is reproducible.

In [ ]:
prompt_base = ("a photograph of a small wooden sailboat on a calm lake, "
               "mountains in the background, highly detailed, sharp focus")

generator = torch.Generator(device=device).manual_seed(10)
init_image = pipe_t2i(prompt_base, num_inference_steps=50, guidance_scale=7.5,
                      generator=generator).images[0]

plt.figure(figsize=(6, 6))
plt.imshow(init_image)
plt.axis("off")
plt.title("starting image (text2img)", fontsize=10)
plt.show()

## Part 1 — img2img

The `StableDiffusionImg2ImgPipeline` takes an **input image** and a **prompt**. It
adds noise to the input (how much is set by `strength` ∈ [0, 1]) and then denoises
guided by the prompt:

- `strength` near **0**: barely any noise added → output ≈ input.
- `strength` near **1**: almost fully noised → behaves like text2img, input ignored.
- intermediate: keeps composition/colours of the input but follows the prompt.

We reuse the weights already loaded — `from_pipe` avoids downloading the model
again.

In [ ]:
pipe_i2i = StableDiffusionImg2ImgPipeline.from_pipe(pipe_t2i).to(device)

prompt_edit = ("a photograph of a small sailboat on a lake at sunset, "
               "warm orange sky, highly detailed, sharp focus")

generator = torch.Generator(device=device).manual_seed(10)
out = pipe_i2i(prompt=prompt_edit, image=init_image, strength=0.6,
               guidance_scale=7.5, generator=generator).images[0]

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(init_image); axes[0].set_title("input", fontsize=11); axes[0].axis("off")
axes[1].imshow(out); axes[1].set_title("img2img (strength=0.6)", fontsize=11); axes[1].axis("off")
plt.tight_layout()
plt.show()

### The `strength` parameter

`strength` is the single most important knob in img2img: it sets how far from the
input the result may drift. We sweep it for the same prompt and seed. Low strength
preserves the original; high strength essentially ignores it. The "sweet spot"
(usually 0.4–0.7) re-styles the image while keeping its structure.

In [ ]:
strengths = [0.2, 0.4, 0.6, 0.8]

fig, axes = plt.subplots(1, len(strengths), figsize=(16, 4.5))
for ax, s in zip(axes, strengths):
    generator = torch.Generator(device=device).manual_seed(10)
    img = pipe_i2i(prompt=prompt_edit, image=init_image, strength=s,
                   guidance_scale=7.5, generator=generator).images[0]
    ax.imshow(img); ax.set_title(f"strength = {s}", fontsize=11); ax.axis("off")
plt.suptitle("img2img: increasing strength drifts further from the input", y=1.02)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

## Part 2 — Inpainting

Inpainting replaces only a **masked region** and leaves the rest of the image
pixel-for-pixel unchanged. It needs a dedicated model that was trained to fill in
masked areas: `stable-diffusion-inpainting`.

The inputs are:
- the **image** to edit,
- a **mask** (white = region to regenerate, black = keep),
- a **prompt** describing what should appear in the masked region.

In [ ]:
from diffusers import StableDiffusionInpaintPipeline

pipe_inpaint = StableDiffusionInpaintPipeline.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-inpainting",
    torch_dtype=torch.float16, safety_checker=None,
).to(device)

print("Inpainting pipeline loaded.")

### Build a mask programmatically

A mask is just a grayscale image: **white (255)** where we want the model to paint,
**black (0)** where the original must be kept. We create a simple rectangular mask
covering the sky region (upper part of the image) — no external tools needed.

In a real application this mask would come from a segmentation model or a user's
brush strokes, but a rectangle makes the mechanism transparent.

In [ ]:
W, H = init_image.size   # SD 1.5 default is 512x512

# White rectangle over the upper third (the sky); black elsewhere.
mask_array = np.zeros((H, W), dtype=np.uint8)
mask_array[: H // 3, :] = 255
mask_image = Image.fromarray(mask_array)

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(init_image); axes[0].set_title("image", fontsize=11); axes[0].axis("off")
axes[1].imshow(mask_image, cmap="gray"); axes[1].set_title("mask (white = repaint)", fontsize=11); axes[1].axis("off")
plt.tight_layout()
plt.show()

### Inpaint the masked region

We now ask the model to repaint **only the sky** (the masked area) with something
new — a dramatic stormy sky — while leaving the boat and lake untouched. Compare
the result to the original: everything below the mask is preserved exactly.

In [ ]:
inpaint_prompt = "a dramatic stormy sky with dark clouds, lightning, highly detailed"

generator = torch.Generator(device=device).manual_seed(5)
result = pipe_inpaint(prompt=inpaint_prompt, image=init_image, mask_image=mask_image,
                      num_inference_steps=50, guidance_scale=7.5,
                      generator=generator).images[0]

fig, axes = plt.subplots(1, 3, figsize=(16, 5.5))
axes[0].imshow(init_image); axes[0].set_title("original", fontsize=11); axes[0].axis("off")
axes[1].imshow(mask_image, cmap="gray"); axes[1].set_title("mask", fontsize=11); axes[1].axis("off")
axes[2].imshow(result); axes[2].set_title("inpainted sky", fontsize=11); axes[2].axis("off")
plt.tight_layout()
plt.show()

## Summary

Both pipelines are the **same diffusion model** from C4 with a different *starting
condition* for the reverse process:

| Pipeline | Starts from | Controlled by | Use case |
|---|---|---|---|
| text2img (C4-6) | pure noise | prompt | generate from scratch |
| **img2img** | noised input image | prompt + `strength` | re-style / vary an image |
| **inpainting** | image + mask | prompt + mask | replace a region |

This is *control through the starting point*. The next notebook (C4-8) adds a
different kind of control — **ControlNet** — which conditions generation on
structural inputs like edges, pose or depth, giving precise spatial control that a
text prompt cannot.

---
## Try It Yourself — Editing with Diffusion

Work in pairs. **Predict first, then run, then explain in one sentence.**

**A. Find your strength.** In the img2img sweep, which `strength` best re-styles
the boat scene while keeping it recognisable? Relate this to C4: what does
`strength` physically control about the diffusion *starting point*?

**B. A different edit.** Change `prompt_edit` to a completely different scene (e.g.
"a sailboat in a tropical lagoon, turquoise water"). At which `strength` does the
input composition finally break down?

**C. Move the mask.** Edit `mask_array` to cover a different region (e.g. the lower
half, the lake). Inpaint it with a matching prompt. Why must the prompt describe
what goes *in the mask*, not the whole scene?

**D. Mask shape matters.** Replace the rectangular mask with a circular one (hint:
use a distance test on a coordinate grid). Does a soft/irregular mask blend better
than a hard rectangle? Why might feathered mask edges help?

**E. Same model, different control.** In one sentence each, explain how text2img,
img2img and inpainting all use the *same* trained denoiser but differ only in the
reverse process's starting condition.